In [ ]:
# !pip install kaggle
# !pip install librosa
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install seaborn
# !pip install tqdm
# !pip install sklearn
# !pip install scipy
# !pip install xgboost
# !pip install lightgbm

In [ ]:
# 📦 Imports
import os
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

# 📁 Paths
DATASET_PATH = r"dataset/dataset"
AUDIO_PATH = os.path.join(DATASET_PATH, "audios_train")  # Assuming your train .wav files are here
TRAIN_CSV = os.path.join(DATASET_PATH, "train.csv")
TEST_CSV = os.path.join(DATASET_PATH, "test.csv")

# 📄 Load CSVs
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# 🧪 Inspect
print("Train samples:", len(train_df))
print("Test samples:", len(test_df))
display(train_df.head())

In [ ]:
import librosa
import numpy as np

def extract_features(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    features = []

    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfccs, axis=1)
    features.extend(mfcc_mean)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(np.mean(chroma, axis=1))

    # Spectral Contrast
    spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    features.extend(np.mean(spec_contrast, axis=1))

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y=y)
    features.append(np.mean(zcr))

    # Tempo (optional)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = librosa.beat.tempo(onset_envelope=onset_env, sr=sr)
    features.append(tempo[0] if len(tempo) > 0 else 0)

    return np.array(features)


In [ ]:
# ✅ Extract features for train and test sets
X_train, y_train = [], []
print("Extracting train features...")

for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    file_name = row["filename"]
    label = row["label"]
    path = os.path.join(AUDIO_PATH, file_name)
    features = extract_features(path)
    X_train.append(features)
    y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

# Extract test features
print("Extracting test features...")
X_test = []
for fname in tqdm(test_df["filename"]):
    path = os.path.join(DATASET_PATH, "audios_test", fname)  # adjust path if needed
    features = extract_features(path)
    X_test.append(features)

X_test = np.array(X_test)


In [ ]:
# ------------------ 📊 Train/Val Split ------------------ #
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# ------------------ 🧪 Define Models and Param Grids ------------------ #
models = {
    "XGBoost": {
        "model": XGBRegressor(tree_method='auto'),
        "params": {
            "n_estimators": [100, 300, 500, 700, 1000],
            "max_depth": [3, 4, 5, 6, 8, 10, 12],
            "learning_rate": [0.001, 0.01, 0.05, 0.07, 0.1, 0.15],
            "subsample": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
            "gamma": [ 0.1, 0.2, 0.3, 0.4],
            "reg_alpha": [ 0.1, 0.5, 1],
            "reg_lambda": [0.5, 1, 1.5, 2]
        }
    },
    "RandomForest": {
        "model": RandomForestRegressor(),
        "params": {
            "n_estimators": [100, 300, 500, 700, 1000],
            "max_depth": [30, 50, 70, 90, 100, None],
            "min_samples_split": [2, 5, 10, 15],
            "min_samples_leaf": [1, 2, 4, 6],
            "max_features": ["sqrt", "log2", None],
            "bootstrap": [True, False]
        }
    },
    "LightGBM": {
        "model": LGBMRegressor(),
        "params": {
            "n_estimators": [100, 300, 500, 700, 1000],
            "learning_rate": [0.001, 0.01, 0.03, 0.05, 0.07, 0.1],
            "num_leaves": [20, 31, 50, 70, 100, 150],
            "max_depth": [-1, 10, 20, 30, 50],
            "min_child_samples": [5, 10, 15, 20, 30],
            "subsample": [0.5, 0.6, 0.7, 0.8, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0],
            "reg_alpha": [ 0.1, 0.5, 1],
            "reg_lambda": [ 1, 2, 3]
        }
    }
}


# ------------------ 🧠 Train, Tune, Evaluate ------------------ #
results = {}

for name, config in models.items():
    print(f"🔍 Tuning {name}...")
    grid = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=40,  # Can be increased to 100+ for more search coverage
        scoring='neg_mean_squared_error',
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42
    )
    grid.fit(X_tr, y_tr)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_val)
    y_pred = np.round(y_pred * 2) / 2
    
    pearson = pearsonr(y_val, y_pred)[0]
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    results[name] = {
        "Pearson": pearson,
        "MAE": mae,
        "RMSE": rmse,
        "BestParams": grid.best_params_
    }

    print(f"\n📊 {name} Results:")
    print(f"  ✅ Best Params: {grid.best_params_}")
    print(f"  📌 Pearson Correlation: {pearson:.4f}")
    print(f"  📌 MAE: {mae:.4f}")
    print(f"  📌 RMSE: {rmse:.4f}\n")

# ------------------  Plot Best Model ------------------ #
best_model_name = max(results, key=lambda k: results[k]["Pearson"])
best_model = models[best_model_name]["model"].set_params(**results[best_model_name]["BestParams"])
best_model.fit(X_tr, y_tr)
y_pred = best_model.predict(X_val)
y_pred = np.clip(np.round(y_pred * 2) / 2, 0.0, 5.0)

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_val, y=y_pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title(f"Best Model: {best_model_name} (Predicted vs Actual)")
plt.grid(True)
plt.show()


In [ ]:
#  Predict on test set
test_preds = best_model.predict(X_test)

# ✍️ Save submission
submission = pd.read_csv(os.path.join(DATASET_PATH, "sample_submission.csv"))
submission["label"] = test_preds
submission.to_csv("submission.csv", index=False)

print("✅ Submission saved to submission.csv")

In [ ]:
search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=50,  # Can be increased to 100+ for more search coverage
        scoring='neg_mean_squared_error',
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42
    )